# Human protein representation workflow

The workflow starts from the reviewed human UniProt table in `data_util/uniprot/` and covers the protein-level representation routes used in this repository.

1. **ESM2 amino-acid sequence**: human protein sequence to ESM2-650M features.
2. **ESMC amino-acid sequence**: the same human FASTA to ESMC-600M features.
3. **GO annotation text**: UniProt accession plus GO text to an API embedding.
4. **STATE protein prior**: export the 5,120-dimensional ESM2-15B table supplied with STATE.
5. **Gene2vec baseline**: load the bundled 200-dimensional pretrained gene vectors.

The routes have different inputs and identifier spaces. Sequence and GO outputs use UniProt accessions. STATE and Gene2vec use gene symbols. They must be aligned through the bundled HGNC mapping before comparison or downstream aggregation.

## 1. Representation routes

| Route | Input field or asset | Encoder | Main output | Identifier |
|---|---|---|---|---|
| ESM2 sequence | `Sequence` in reviewed human UniProt | ESM2-650M | pooled `.npy` or all-token `.npz` | UniProt `Entry` |
| ESMC sequence | `Sequence` in reviewed human UniProt | ESMC-600M | pooled `.npy` or all-token `.npz` | UniProt `Entry` |
| GO text | `Entry` and `Gene Ontology (GO)` | OpenAI or Qwen text embedding model | `.npz` with `ids` and `embeddings` | UniProt `Entry` |
| STATE | `protein_embeddings.pt` from SE-600M | precomputed ESM2-15B prior used by STATE | gene-by-feature `.h5ad` | HGNC symbol |
| Gene2vec | bundled pretrained NPZ | Gene2vec | gene-by-feature array | gene symbol |

ESM2 and ESMC compute embeddings directly from amino-acid sequences. GO embeddings describe annotation text. The STATE route exports a released model asset; it does not rerun ESM2-15B on the bundled human sequences. Gene2vec is a pretrained gene baseline rather than a protein sequence model.

## 2. Environment

The sequence workflow was developed with Python 3.10, PyTorch 2.4.1, Transformers 4.37.0 and CUDA 11.8. Install the PyTorch build that matches the local driver. The STATE export script only reads a PyTorch dictionary and writes AnnData; the full STATE training environment is not required.

```bash
conda create -n protein-representation python=3.10 -y
conda activate protein-representation
conda install pytorch=2.4.1 torchvision=0.19.1 torchaudio=2.4.1 pytorch-cuda=11.8 -c pytorch -c nvidia
python -m pip install transformers==4.37.0 tokenizers==0.15.2 accelerate biopython pandas openpyxl numpy tqdm anndata huggingface-hub openai jupyter esm
```

For CPU-only use, install the CPU PyTorch build instead. Full-human ESM2 extraction is substantially faster on a CUDA GPU.

In [ ]:
from pathlib import Path
import os
import subprocess

import numpy as np
import pandas as pd

# Find the repository root whether Jupyter starts in the root or tutorials/.
ROOT = Path.cwd().resolve()
if not (ROOT / "protein-representation-generation").exists():
    ROOT = ROOT.parent
assert (ROOT / "protein-representation-generation").exists(), "Start Jupyter from the repository root or tutorials directory."

HUMAN_XLSX = ROOT / "data_util/uniprot/uniprotkb_human_reviewed_2025-02-28.xlsx"
HGNC_TSV = ROOT / "data_util/mappings/hgnc_complete_set.txt"
GENE2VEC_NPZ = ROOT / "data_util/embeddings/gene2vec/gene2vec_dim_200_iter_9.npz"
WORK_DIR = ROOT / "outputs/tutorial-human-protein-representations"
FASTA_DIR = WORK_DIR / "fasta"
GO_INPUT_DIR = WORK_DIR / "go-input"
MODEL_DIR = ROOT / "models"

for directory in (WORK_DIR, FASTA_DIR, GO_INPUT_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## 3. Inspect and prepare the reviewed human proteins

The bundled workbook contains 18 columns. The fields used here are `Entry`, `Gene Names`, `Sequence` and `Gene Ontology (GO)`. `Entry` is retained as the stable protein identifier. The first token in `Gene Names` can be used as a candidate gene symbol, but HGNC mapping remains the authoritative alignment route.

In [ ]:
required_columns = ["Entry", "Gene Names", "Sequence", "Gene Ontology (GO)"]
human = pd.read_excel(HUMAN_XLSX, dtype=str, usecols=required_columns, engine="openpyxl")

missing = set(required_columns) - set(human.columns)
assert not missing, f"Missing UniProt columns: {sorted(missing)}"

human = human.dropna(subset=["Entry", "Sequence"]).copy()
human["Entry"] = human["Entry"].str.strip()
human["Sequence"] = human["Sequence"].str.replace(r"\s+", "", regex=True).str.upper()
human["primary_gene_symbol"] = human["Gene Names"].fillna("").str.split().str[0]

assert human["Entry"].is_unique, "UniProt accessions must be unique."
assert human["Sequence"].str.len().gt(0).all(), "Empty sequences remain after filtering."

human[["Entry", "primary_gene_symbol", "Sequence", "Gene Ontology (GO)"]].head()

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

fasta_path = FASTA_DIR / "human_reviewed_2025-02-28.fasta"
records = []
for row in human.itertuples(index=False):
    accession = row.Entry
    sequence = row.Sequence
    gene = row.primary_gene_symbol if pd.notna(row.primary_gene_symbol) and row.primary_gene_symbol else "NA"
    records.append(SeqRecord(Seq(sequence), id=accession, description=f"gene={gene}"))

SeqIO.write(records, fasta_path, "fasta")
print(f"Prepared {len(records):,} FASTA records at {fasta_path}")

## 4. Download model assets

The ESM2 script accepts either the Hugging Face model name or a local checkpoint directory. ESMC uses a local weight file and tokenizer directory. The STATE file is downloaded separately because it is already a gene-indexed protein representation table.

Set the required download flags to `True` before running the next cell.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

DOWNLOAD_ESM2 = False
DOWNLOAD_ESMC = False
DOWNLOAD_STATE = False

ESM2_MODEL_ID = "facebook/esm2_t33_650M_UR50D"
ESM2_LOCAL_DIR = MODEL_DIR / "esm2_t33_650M_UR50D"
ESMC_MODEL_ID = "EvolutionaryScale/esmc-600m-2024-12"
ESMC_LOCAL_DIR = MODEL_DIR / "esmc-600m-2024-12"
ESMC_TOKENIZER_ID = "facebook/esm2_t6_8M_UR50D"
ESMC_TOKENIZER_DIR = MODEL_DIR / "esm2_t6_8M_UR50D"
ESMC_WEIGHTS = ESMC_LOCAL_DIR / "data/weights/esmc_600m_2024_12_v0.pth"
STATE_LOCAL_DIR = MODEL_DIR / "SE-600M"

if DOWNLOAD_ESM2:
    snapshot_download(repo_id=ESM2_MODEL_ID, local_dir=ESM2_LOCAL_DIR)

if DOWNLOAD_ESMC:
    snapshot_download(repo_id=ESMC_MODEL_ID, local_dir=ESMC_LOCAL_DIR)
    snapshot_download(repo_id=ESMC_TOKENIZER_ID, local_dir=ESMC_TOKENIZER_DIR)

if DOWNLOAD_STATE:
    hf_hub_download(
        repo_id="arcinstitute/SE-600M",
        filename="protein_embeddings.pt",
        local_dir=STATE_LOCAL_DIR,
    )

## 5. Sequence embedding with ESM2-650M

The pooled entry script uses the final hidden layer and concatenates CLS, EOS, the mean over sequence residues and ten segment means. ESM2-650M has a hidden width of 1,280, so the resulting matrix has `13 × 1,280 = 16,640` columns when every feature group is enabled.

The script retains sequences shorter than 4,024 residues and writes three files for each FASTA input: the filtered sequence table, a numerical `.npy` matrix and an Excel table listing the output columns. Set `RUN_ESM2` to `True` to execute the command.

In [ ]:
RUN_ESM2 = False
ESM2_OUTPUT = WORK_DIR / "esm2-650m-pooled"
ESM2_OUTPUT.mkdir(parents=True, exist_ok=True)

# Use ESM2_LOCAL_DIR after downloading, or ESM2_MODEL_ID for direct Hub loading.
esm2_source = str(ESM2_LOCAL_DIR) if ESM2_LOCAL_DIR.exists() else ESM2_MODEL_ID
esm2_script = ROOT / "protein-representation-generation/scripts/sequence/ESM2_650M_feature_extraction_fasta.py"

esm2_command = [
    "python", str(esm2_script),
    "--model_path", esm2_source,
    "--open_path", str(FASTA_DIR),
    "--save_path", str(ESM2_OUTPUT),
    "--device", "cuda:0",
    "--batch_size", "1",
]
print(" ".join(esm2_command))
if RUN_ESM2:
    subprocess.run(esm2_command, check=True)

### Extract CLS, EOS, mean and segment blocks separately

The pooled `.npy` matrix stores the 13 blocks in this order:

| Block | Columns in the pooled matrix | Shape per protein |
|---|---:|---:|
| `cls` | `0:1280` | 1,280 |
| `eos` | `1280:2560` | 1,280 |
| `mean` | `2560:3840` | 1,280 |
| `seg0` | `3840:5120` | 1,280 |
| `seg1`–`seg9` | successive 1,280-column blocks | 1,280 each |

For segment `k`, use columns `[(3 + k) × 1280 : (4 + k) × 1280]`. Thus, `seg9` occupies `15360:16640`. The code below checks these positions against the feature names written by the extraction script before returning or saving the separate matrices.

In [ ]:
pooled_matrix_path = ESM2_OUTPUT / "human_reviewed_2025-02-28.npy"
feature_columns_path = ESM2_OUTPUT / "human_reviewed_2025-02-28_feature_columns.xlsx"
SAVE_SEPARATE_BLOCKS = False

if pooled_matrix_path.exists() and feature_columns_path.exists():
    pooled_matrix = np.load(pooled_matrix_path, mmap_mode="r")
    feature_table = pd.read_excel(feature_columns_path, dtype=str)
    all_feature_names = feature_table["Feature_Columns"].dropna().tolist()
    embedding_feature_names = [name for name in all_feature_names if name.startswith("ESM2_")]

    hidden_dim = 1280
    block_slices = {
        "cls": slice(0, hidden_dim),
        "eos": slice(hidden_dim, 2 * hidden_dim),
        "mean": slice(2 * hidden_dim, 3 * hidden_dim),
    }
    block_slices.update({
        f"seg{k}": slice((3 + k) * hidden_dim, (4 + k) * hidden_dim)
        for k in range(10)
    })

    expected_prefixes = {
        "cls": "ESM2_cls",
        "eos": "ESM2_eos",
        "mean": "ESM2_mean",
        **{f"seg{k}": f"ESM2_segment{k}_mean" for k in range(10)},
    }

    assert pooled_matrix.shape[1] == 13 * hidden_dim
    assert len(embedding_feature_names) == pooled_matrix.shape[1]
    for block_name, block_slice in block_slices.items():
        names = embedding_feature_names[block_slice]
        assert len(names) == hidden_dim
        assert all(name.startswith(expected_prefixes[block_name]) for name in names)

    pooled_blocks = {
        block_name: np.asarray(pooled_matrix[:, block_slice])
        for block_name, block_slice in block_slices.items()
    }

    cls_embedding = pooled_blocks["cls"]
    eos_embedding = pooled_blocks["eos"]
    mean_embedding = pooled_blocks["mean"]
    seg0_embedding = pooled_blocks["seg0"]

    for block_name, block_matrix in pooled_blocks.items():
        print(block_name, block_matrix.shape)
        if SAVE_SEPARATE_BLOCKS:
            np.save(ESM2_OUTPUT / f"human_reviewed_2025-02-28_{block_name}.npy", block_matrix)
else:
    print("Run the pooled ESM2 extraction first to create the matrix and feature-column table.")

### Optional all-token output

Use the all-token entry script when residue-level vectors are required. It writes a compressed NPZ mapping each UniProt accession to a variable-length token matrix. The arrays include model special tokens, so do not treat their first and last rows as amino-acid residues.

In [ ]:
RUN_ESM2_ALL_TOKEN = False
all_token_script = ROOT / "protein-representation-generation/scripts/sequence/ESM2_650M_all_token_feature_extraction_fasta.py"
all_token_output = WORK_DIR / "esm2-650m-all-token"
all_token_output.mkdir(parents=True, exist_ok=True)
all_token_command = [
    "python", str(all_token_script),
    "--model_path", esm2_source,
    "--open_path", str(FASTA_DIR),
    "--save_path", str(all_token_output),
    "--device", "cuda:0",
    "--batch_size", "1",
]
print(" ".join(all_token_command))
if RUN_ESM2_ALL_TOKEN:
    subprocess.run(all_token_command, check=True)

## 6. Sequence embedding with ESMC-600M

ESMC uses the same human FASTA input but loads `esmc_600m_2024_12_v0.pth` through the `esm` package. The repository implementation has `d_model = 1,152`, 18 attention heads and 36 layers. Its pooled extractor concatenates CLS, EOS, residue mean and ten segment means in the same order as the ESM2 route. The code-derived output width is therefore `13 × 1,152 = 14,976`.

The checkpoint snapshot and ESM2-8M tokenizer are configured in the model-download cell above. Set `RUN_ESMC` to `True` after checking both local paths.

In [ ]:
RUN_ESMC = False
ESMC_OUTPUT = WORK_DIR / "esmc-600m-pooled"
ESMC_OUTPUT.mkdir(parents=True, exist_ok=True)
esmc_script = ROOT / "protein-representation-generation/scripts/sequence/ESMC_600M_feature_extraction_fasta.py"

esmc_command = [
    "python", str(esmc_script),
    "--weights_path", str(ESMC_WEIGHTS),
    "--tokenizer_path", str(ESMC_TOKENIZER_DIR),
    "--open_path", str(FASTA_DIR),
    "--save_path", str(ESMC_OUTPUT),
    "--device", "cuda:0",
    "--max_length", "4024",
]
print(" ".join(esmc_command))
if RUN_ESMC:
    assert ESMC_WEIGHTS.exists(), "Download or place the ESMC weight file first."
    assert ESMC_TOKENIZER_DIR.exists(), "Download the tokenizer first."
    subprocess.run(esmc_command, check=True)

### Extract ESMC CLS, EOS, mean and segment blocks separately

For ESMC, `cls`, `eos` and `mean` occupy the first three 1,152-column blocks. Segment `k` uses columns `[(3 + k) × 1152 : (4 + k) × 1152]`; `seg0` is `3456:4608` and `seg9` is `13824:14976`. The feature-column table is checked before the blocks are returned or saved.

In [ ]:
esmc_matrix_path = ESMC_OUTPUT / "human_reviewed_2025-02-28.npy"
esmc_columns_path = ESMC_OUTPUT / "human_reviewed_2025-02-28_feature_columns.xlsx"
SAVE_ESMC_BLOCKS = False

if esmc_matrix_path.exists() and esmc_columns_path.exists():
    esmc_matrix = np.load(esmc_matrix_path, mmap_mode="r")
    esmc_feature_table = pd.read_excel(esmc_columns_path, dtype=str)
    esmc_all_names = esmc_feature_table["Feature_Columns"].dropna().tolist()
    esmc_feature_names = [name for name in esmc_all_names if name.startswith("ESMC_")]

    esmc_hidden_dim = 1152
    esmc_block_slices = {
        "cls": slice(0, esmc_hidden_dim),
        "eos": slice(esmc_hidden_dim, 2 * esmc_hidden_dim),
        "mean": slice(2 * esmc_hidden_dim, 3 * esmc_hidden_dim),
        **{
            f"seg{k}": slice((3 + k) * esmc_hidden_dim, (4 + k) * esmc_hidden_dim)
            for k in range(10)
        },
    }
    esmc_prefixes = {
        "cls": "ESMC_cls",
        "eos": "ESMC_eos",
        "mean": "ESMC_mean",
        **{f"seg{k}": f"ESMC_segment{k}_mean" for k in range(10)},
    }

    assert esmc_matrix.shape[1] == 13 * esmc_hidden_dim
    assert len(esmc_feature_names) == esmc_matrix.shape[1]
    for block_name, block_slice in esmc_block_slices.items():
        names = esmc_feature_names[block_slice]
        assert len(names) == esmc_hidden_dim
        assert all(name.startswith(esmc_prefixes[block_name]) for name in names)

    esmc_blocks = {
        block_name: np.asarray(esmc_matrix[:, block_slice])
        for block_name, block_slice in esmc_block_slices.items()
    }
    esmc_cls_embedding = esmc_blocks["cls"]
    esmc_eos_embedding = esmc_blocks["eos"]
    esmc_mean_embedding = esmc_blocks["mean"]
    esmc_seg0_embedding = esmc_blocks["seg0"]

    for block_name, block_matrix in esmc_blocks.items():
        print(block_name, block_matrix.shape)
        if SAVE_ESMC_BLOCKS:
            np.save(ESMC_OUTPUT / f"human_reviewed_2025-02-28_{block_name}.npy", block_matrix)
else:
    print("Run the pooled ESMC extraction first to create the matrix and feature-column table.")

### Optional ESMC all-token output

The all-token script stores one variable-length array per UniProt accession in a compressed NPZ file. These arrays include special tokens.

In [ ]:
RUN_ESMC_ALL_TOKEN = False
esmc_all_token_output = WORK_DIR / "esmc-600m-all-token"
esmc_all_token_output.mkdir(parents=True, exist_ok=True)
esmc_all_token_script = ROOT / "protein-representation-generation/scripts/sequence/ESMC_600M_all_token_feature_extraction_fasta.py"
esmc_all_token_command = [
    "python", str(esmc_all_token_script),
    "--weights_path", str(ESMC_WEIGHTS),
    "--tokenizer_path", str(ESMC_TOKENIZER_DIR),
    "--open_path", str(FASTA_DIR),
    "--save_path", str(esmc_all_token_output),
    "--device", "cuda:0",
    "--max_length", "4024",
]
print(" ".join(esmc_all_token_command))
if RUN_ESMC_ALL_TOKEN:
    assert ESMC_WEIGHTS.exists() and ESMC_TOKENIZER_DIR.exists()
    subprocess.run(esmc_all_token_command, check=True)

## 7. GO annotation text embedding

This route does not encode amino-acid sequences. The preprocessing script creates a JSON mapping from UniProt `Entry` to `Gene Ontology (GO)`. The embedding scripts construct text in the form `Uniprot ID: <accession>; <GO annotation>` and preserve the JSON order in the NPZ `ids` array.

The first cell below isolates the human workbook so the directory-based converter does not process the Archaea, Bacteria and Eukaryota snapshots.

In [ ]:
go_workbook = GO_INPUT_DIR / "human_uniprot_go.xlsx"
human[["Entry", "Gene Ontology (GO)"]].to_excel(go_workbook, index=False)

GO_JSON_DIR = WORK_DIR / "go-json"
go_converter = ROOT / "protein-representation-generation/scripts/go_text/step1_protein_GO_JSON_preprocessed.py"
subprocess.run([
    "python", str(go_converter),
    "--open_path", str(GO_INPUT_DIR),
    "--save_path", str(GO_JSON_DIR),
    "--protein_ID", "Entry",
    "--GO_annotation", "Gene Ontology (GO)",
], check=True)

In [ ]:
RUN_OPENAI_GO = False
RUN_QWEN_GO = False
GO_EMBED_OUTPUT = WORK_DIR / "go-embeddings"
GO_EMBED_OUTPUT.mkdir(parents=True, exist_ok=True)

if RUN_OPENAI_GO:
    api_key = os.environ["OPENAI_API_KEY"]
    subprocess.run([
        "python", str(ROOT / "protein-representation-generation/scripts/go_text/step2_protein_GPT_embedding_batch.py"),
        "--open_path", str(GO_JSON_DIR),
        "--save_path", str(GO_EMBED_OUTPUT / "openai"),
        "--save_format", "npz",
        "--model", "text-embedding-3-small",
        "--dim", "1024",
        "--api_key", api_key,
    ], check=True)

if RUN_QWEN_GO:
    api_key = os.environ["DASHSCOPE_API_KEY"]
    subprocess.run([
        "python", str(ROOT / "protein-representation-generation/scripts/go_text/step2_protein_Qwen_embedding_batch.py"),
        "--open_path", str(GO_JSON_DIR),
        "--save_path", str(GO_EMBED_OUTPUT / "qwen"),
        "--save_format", "npz",
        "--dim", "1024",
        "--api_key", api_key,
    ], check=True)

## 8. Export the STATE ESM2-15B protein prior

`protein_embeddings.pt` is distributed with Arc Institute SE-600M. It maps gene symbols to 5,120-dimensional vectors from the ESM2-15B protein prior used by the STATE tokenizer. The repository script sorts the gene symbols, stacks the vectors and writes `STATE_tok_5120_Emb.h5ad`, `gene_list.txt` and `gene_embedding_metadata.md`.

This is an export step, not sequence inference. Set `RUN_STATE_EXPORT` to `True` after downloading the asset.

In [ ]:
RUN_STATE_EXPORT = False
STATE_PT = STATE_LOCAL_DIR / "protein_embeddings.pt"
STATE_OUTPUT = WORK_DIR / "state-esm2-15b"
state_script = ROOT / "protein-representation-generation/scripts/state/STATE_tok_5120_Emb_extract.py"

state_command = [
    "python", str(state_script),
    "--open_path", str(STATE_PT),
    "--save_path", str(STATE_OUTPUT),
]
print(" ".join(state_command))
if RUN_STATE_EXPORT:
    assert STATE_PT.exists(), "Download protein_embeddings.pt first."
    subprocess.run(state_command, check=True)

## 9. Load the Gene2vec baseline

The bundled Gene2vec NPZ is a pretrained gene representation baseline. It contains 24,447 entries in `gene_names` and a `float32` `embeddings` matrix with 200 columns. Loading this file is not protein sequence inference and does not use the UniProt amino-acid sequence.

Use exact gene-symbol matching first. Any alias or previous-symbol recovery should be performed explicitly with HGNC fields rather than silently changing identifiers.

In [ ]:
gene2vec_data = np.load(GENE2VEC_NPZ, allow_pickle=False)
assert set(gene2vec_data.files) == {"gene_names", "embeddings"}

gene2vec_genes = gene2vec_data["gene_names"].astype(str)
gene2vec_embeddings = gene2vec_data["embeddings"]
assert gene2vec_embeddings.dtype == np.float32
assert gene2vec_embeddings.shape == (len(gene2vec_genes), 200)
assert len(set(gene2vec_genes.tolist())) == len(gene2vec_genes)

gene2vec_row = pd.Series(np.arange(len(gene2vec_genes)), index=gene2vec_genes, name="row")
example_symbols = [symbol for symbol in ["TP53", "EGFR", "CD3D"] if symbol in gene2vec_row.index]
example_vectors = {
    symbol: gene2vec_embeddings[gene2vec_row.loc[symbol]]
    for symbol in example_symbols
}
print("Gene2vec:", gene2vec_embeddings.shape, "examples:", example_symbols)

## 10. Align UniProt accessions, STATE and Gene2vec

The HGNC table stores approved gene symbols in `symbol` and one or more UniProt accessions in `uniprot_ids`. Multiple accessions are separated by `|`. Explode this field before joining sequence or GO rows to STATE rows. Gene2vec names can be checked directly against approved HGNC symbols. Preserve the original UniProt order and decide explicitly how multiple proteins for one gene are handled; do not silently average isoforms or accessions.

In [ ]:
hgnc_full = pd.read_csv(HGNC_TSV, sep="\t", dtype=str, usecols=["symbol", "uniprot_ids"])
hgnc = hgnc_full.dropna(subset=["symbol", "uniprot_ids"]).copy()
hgnc["Entry"] = hgnc["uniprot_ids"].str.split("|")
hgnc = hgnc.explode("Entry", ignore_index=True)
hgnc["Entry"] = hgnc["Entry"].str.strip()

human_to_hgnc = human[["Entry", "Sequence", "Gene Ontology (GO)"]].merge(
    hgnc[["symbol", "Entry"]], on="Entry", how="left", validate="one_to_many"
)
gene2vec_approved = pd.Series(gene2vec_genes).isin(set(hgnc_full["symbol"].dropna()))
print("Exact Gene2vec-to-HGNC coverage:", f"{gene2vec_approved.mean():.1%}")
human_to_hgnc.head()

## 11. Validate generated outputs

Run these checks after the selected extraction routes finish. The checks verify row counts, dimensions, identifier uniqueness and mapping coverage without loading the complete sequence matrix into memory.

In [ ]:
import anndata as ad

sequence_matrix_path = ESM2_OUTPUT / "human_reviewed_2025-02-28.npy"
sequence_rows_path = ESM2_OUTPUT / "human_reviewed_2025-02-28_filtered.xlsx"

if sequence_matrix_path.exists():
    sequence_matrix = np.load(sequence_matrix_path, mmap_mode="r")
    sequence_rows = pd.read_excel(sequence_rows_path, dtype=str)
    assert sequence_matrix.shape[0] == len(sequence_rows)
    assert sequence_matrix.shape[1] == 16640
    assert sequence_rows["ID"].is_unique
    print("ESM2 pooled:", sequence_matrix.shape)

esmc_validation_path = ESMC_OUTPUT / "human_reviewed_2025-02-28.npy"
esmc_rows_path = ESMC_OUTPUT / "human_reviewed_2025-02-28_filtered.xlsx"
if esmc_validation_path.exists():
    esmc_validation_matrix = np.load(esmc_validation_path, mmap_mode="r")
    esmc_rows = pd.read_excel(esmc_rows_path, dtype=str)
    assert esmc_validation_matrix.shape[0] == len(esmc_rows)
    assert esmc_validation_matrix.shape[1] == 14976
    assert esmc_rows["ID"].is_unique
    print("ESMC pooled:", esmc_validation_matrix.shape)

go_npz = GO_EMBED_OUTPUT / "openai/human_uniprot_go.npz"
if go_npz.exists():
    go_data = np.load(go_npz, allow_pickle=False)
    assert len(go_data["ids"]) == go_data["embeddings"].shape[0]
    assert len(set(go_data["ids"].tolist())) == len(go_data["ids"])
    print("GO text:", go_data["embeddings"].shape)

state_h5ad = STATE_OUTPUT / "STATE_tok_5120_Emb.h5ad"
if state_h5ad.exists():
    state = ad.read_h5ad(state_h5ad, backed="r")
    assert state.n_vars == 5120
    assert state.obs_names.is_unique
    mapped = human_to_hgnc["symbol"].isin(state.obs_names).mean()
    print("STATE:", state.shape, "mapped human rows:", f"{mapped:.1%}")
    state.file.close()

assert gene2vec_embeddings.shape[1] == 200
assert gene2vec_embeddings.shape[0] == len(gene2vec_genes)
print("Gene2vec:", gene2vec_embeddings.shape)

## 12. Record with each output

Keep the following information with each representation matrix:

- UniProt snapshot filename and review status.
- Model repository and revision or local checkpoint identity.
- Pooling fields and sequence-length filter.
- Ordered identifier file and matrix shape.
- GO text model, requested dimension and API endpoint when applicable.
- Gene2vec NPZ filename and ordered gene-name array.
- HGNC mapping filename used for accession-to-symbol alignment.

Do not treat GO-text vectors, the STATE export or Gene2vec as newly calculated amino-acid sequence embeddings.